# 🖊️ Task 1: Handwritten Text Generation
## Character-Level RNN (LSTM) – CodSoft AI/ML Internship

**Google Colab Edition**: This notebook is optimized exclusively for Google Colab. It mounts your Google Drive to ensure your trained models and generated outputs are securely saved and can be accessed even after your Colab session ends.

## 💾 Step 1: Mount Google Drive & Setup Directories

In [ ]:
from google.colab import drive
import os

# Mount Google Drive to save the model and outputs permanently
drive.mount('/content/drive')

# Define paths on Google Drive for saving models and outputs
SAVE_DIR = '/content/drive/MyDrive/CodSoft_Task1'
MODELS_DIR = os.path.join(SAVE_DIR, 'models')
OUTPUTS_DIR = os.path.join(SAVE_DIR, 'outputs')

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(OUTPUTS_DIR, exist_ok=True)
print(f'✅ Models will be saved to: {MODELS_DIR}')
print(f'✅ Outputs will be saved to: {OUTPUTS_DIR}')

## 🔑 Step 2: Kaggle Credentials & Dataset Download

In [ ]:
from google.colab import files
import shutil

# Upload kaggle.json if it's not already in place
if not os.path.exists('/root/.kaggle/kaggle.json'):
    print('⚠️ Please upload your kaggle.json file (downloaded from Kaggle Account Settings)')
    uploaded = files.upload()
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    shutil.move('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
    os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

DATA_DIR = '/content/dataset'
CSV_PATH = os.path.join(DATA_DIR, 'english.csv')
IMG_DIR  = os.path.join(DATA_DIR, 'Img')

if not os.path.exists(CSV_PATH):
    print('📥 Downloading dataset from Kaggle...')
    !pip install kaggle -q
    os.makedirs(DATA_DIR, exist_ok=True)
    !kaggle datasets download -d dhruvildave/english-handwritten-characters-dataset -p {DATA_DIR} --unzip
    print('✅ Dataset downloaded and unzipped!')
else:
    print('✅ Dataset already exists.')

## 📦 Step 3: Install Dependencies & Import Libraries

In [ ]:
!pip install tensorflow numpy pandas matplotlib Pillow opencv-python scikit-learn seaborn tqdm -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import random
import json
import warnings
warnings.filterwarnings('ignore')

from PIL import Image
from tqdm import tqdm
from collections import Counter

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

# Ensure GPU is used
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f'✅ GPU found: {[g.name for g in gpus]}')
else:
    print('❌ No GPU found - Please go to Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU')
print(f'TensorFlow version: {tf.__version__}')

## 📊 Step 4: Load Dataset & EDA

In [ ]:
df = pd.read_csv(CSV_PATH)
if len(df.columns) == 2:
    df.columns = ['image', 'label']
elif 'label' not in df.columns:
    df.rename(columns={df.columns[-1]: 'label', df.columns[0]: 'image'}, inplace=True)

print(f'Total samples: {len(df)}')
print(f'Unique characters: {df["label"].nunique()}')
print(df.head())

fig, axes = plt.subplots(1, 2, figsize=(18, 5))
label_counts = df['label'].value_counts().sort_index()
axes[0].bar(range(len(label_counts)), label_counts.values, color='steelblue')
axes[0].set_xticks(range(len(label_counts)))
axes[0].set_xticklabels(label_counts.index, fontsize=7)
axes[0].set_title('Character Label Distribution', fontsize=14)

labels_list = label_counts.index.tolist()
digits = [l for l in labels_list if str(l).isdigit()]
upper  = [l for l in labels_list if str(l).isupper()]
lower  = [l for l in labels_list if str(l).islower()]
axes[1].pie([len(digits), len(upper), len(lower)], labels=[f'Digits', f'Uppercase', f'Lowercase'], autopct='%1.1f%%')
axes[1].set_title('Character Categories', fontsize=14)
plt.savefig(os.path.join(OUTPUTS_DIR, 'eda.png'))
plt.show()

## 🖼️ Step 5: Build Image Bank (For Rendering Generated Text)

In [ ]:
def load_image(img_filename, img_dir, size=(64, 64)):
    path = os.path.join(img_dir, img_filename)
    if not os.path.exists(path): return None
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None: return None
    return cv2.resize(img, size)

image_bank = {}
for _, row in tqdm(df.iterrows(), total=len(df), desc='Building image bank'):
    char = str(row['label'])
    fname = row['image']
    if char not in image_bank: image_bank[char] = []
    image_bank[char].append(fname)
print(f'✅ Image bank ready with {len(image_bank)} characters')

## 🔤 Step 6: Text Processing & Sequences Preparation

In [ ]:
corpus = list(df['label'].astype(str).values)
vocab = sorted(set(corpus))
vocab_size = len(vocab)
char2idx = {c: i for i, c in enumerate(vocab)}
idx2char = {i: c for c, i in char2idx.items()}
corpus_encoded = [char2idx[c] for c in corpus]

SEQ_LENGTH = 40
STEP = 3
BATCH_SIZE = 512 # Colab T4 can handle 512
EPOCHS = 40

X_seqs, y_seqs = [], []
for i in range(0, len(corpus_encoded) - SEQ_LENGTH, STEP):
    X_seqs.append(corpus_encoded[i : i + SEQ_LENGTH])
    y_seqs.append(corpus_encoded[i + SEQ_LENGTH])

X_seqs = np.array(X_seqs)
y_seqs = np.array(y_seqs)
y_onehot = to_categorical(y_seqs, num_classes=vocab_size)
X_train, X_val, y_train, y_val = train_test_split(X_seqs, y_onehot, test_size=0.1, random_state=42)

print(f'Train shapes: X={X_train.shape}, y={y_train.shape}')

## 🧠 Step 7: Define the LSTM Model

In [ ]:
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=64, input_length=SEQ_LENGTH),
    LSTM(256, return_sequences=True),
    Dropout(0.3),
    LSTM(256, return_sequences=True),
    Dropout(0.3),
    LSTM(128, return_sequences=False),
    Dropout(0.2),
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dense(vocab_size, activation='softmax')
])

model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

## 🚀 Step 8: Train & Save the Model

In [ ]:
best_model_path = os.path.join(MODELS_DIR, 'char_rnn_best.h5')

callbacks = [
    ModelCheckpoint(best_model_path, monitor='val_accuracy', save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6)
]

history = model.fit(
    X_train, y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(X_val, y_val),
    callbacks=callbacks
)

# Save final model as well
final_model_path = os.path.join(MODELS_DIR, 'char_rnn_final.h5')
model.save(final_model_path)
print(f'✅ Final Model saved to Google Drive: {final_model_path}')

# Save vocabulary
vocab_data = {'char2idx': char2idx, 'idx2char': {str(k): v for k, v in idx2char.items()}, 'seq_length': SEQ_LENGTH, 'vocab_size': vocab_size}
vocab_path = os.path.join(MODELS_DIR, 'vocab.json')
with open(vocab_path, 'w') as f:
    json.dump(vocab_data, f)
print(f'✅ Vocabulary saved to Google Drive: {vocab_path}')

## ✨ Step 9: Text Generation (With Temperature Sampling)

In [ ]:
def sample_with_temperature(predictions, temperature=1.0):
    predictions = np.asarray(predictions).astype('float64')
    predictions = np.log(predictions + 1e-8) / temperature
    exp_preds = np.exp(predictions - np.max(predictions))
    predictions = exp_preds / np.sum(exp_preds)
    return np.argmax(np.random.multinomial(1, predictions, 1))

def generate_text(model, seed_chars, num_generate=100, temperature=1.0):
    seed = [c for c in seed_chars if c in char2idx]
    if len(seed) < SEQ_LENGTH:
        seed = [random.choice(list(char2idx.keys())) for _ in range(SEQ_LENGTH - len(seed))] + seed
    seed = seed[-SEQ_LENGTH:]

    generated = list(seed)
    current_seq = [char2idx[c] for c in seed]

    for _ in range(num_generate):
        x = np.array(current_seq[-SEQ_LENGTH:]).reshape(1, SEQ_LENGTH)
        preds = model.predict(x, verbose=0)[0]
        next_idx = sample_with_temperature(preds, temperature)
        generated.append(idx2char[next_idx])
        current_seq.append(next_idx)

    return generated[SEQ_LENGTH:]

seed_sequence = corpus[:SEQ_LENGTH]
print(f"Seed: {''.join(seed_sequence)}\n")
for temp in [0.5, 1.0, 1.5]:
    gen = generate_text(model, seed_sequence, num_generate=80, temperature=temp)
    print(f'T={temp}: {''.join(gen)}')

## 🖼️ Step 10: Render Generated Text as Handwritten Images

In [ ]:
def render_handwritten(char_sequence, char_size=64, padding=4):
    char_images = []
    for char in char_sequence:
        if char in image_bank and image_bank[char]:
            img = load_image(random.choice(image_bank[char]), IMG_DIR, size=(char_size, char_size))
            char_images.append(img if img is not None else np.full((char_size, char_size), 255, dtype=np.uint8))
        else:
            char_images.append(np.full((char_size, char_size), 255, dtype=np.uint8))
    if not char_images: return None
    pad = np.full((char_size, padding), 255, dtype=np.uint8)
    row = char_images[0]
    for img in char_images[1:]: row = np.concatenate([row, pad, img], axis=1)
    return row

gen_seq = generate_text(model, seed_sequence, num_generate=30, temperature=1.0)
rendered = render_handwritten(gen_seq)
plt.figure(figsize=(15, 3))
plt.imshow(rendered, cmap='gray')
plt.title(f"Generated: {''.join(gen_seq)}")
plt.axis('off')
plt.savefig(os.path.join(OUTPUTS_DIR, 'handwritten_output.png'))
plt.show()